# Candidate Recovery Metric — Resting Heart Rate

Resting heart rate is the wearable signal least dependent on a patient choosing to be active, so
**it can register recovery in someone too unwell to walk — exactly the patients a step-count outcome
loses**. It is also the hardest to obtain: All of Us publishes daily heart-rate zone summaries but no
resting value, so the metric has to be derived from the minute-level table, and cost is therefore a
real selection criterion here rather than an afterthought.

**Every reading of the signal is within patient.** A per-day median over whoever wore the device is
cross-sectional: it moves when the set of wearers moves, so a peri-operative rise in one is as
consistent with the fittest patients taking the device off as with anyone's heart working harder.
Each day is therefore read against that patient's own normal, using the four baselines in
`baselines.py` — a frozen pre-operative lookback, a trailing flat window, a trailing exponential
window, and a short window read against a long one. The cross-sectional curve appears once, dotted,
as the context the baselines exist to replace.

**Coverage is settled from the cheap table and the signal from the expensive one, in that order.**
Nothing touches `heart_rate_intraday` until a dry run has priced the scan and it comes in under the
budget set in the first cell; if it does not, the notebook says so and stops, because the cost is
itself an argument against the metric.

**Governance.** The derived person-day extract is participant-level and stays in the kernel; every
printed number and every figure comes through `srp_common.reportable_days`, which drops any day
fewer than `SUPPRESS_THRESHOLD` patients contributed to. Unsuppressed figures stay git-ignored while
`*_suppressed.png` is committable, and the threshold must be confirmed against current All of Us
policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR, creates this notebook's git-ignored figure and cache
directories, and builds the cohort plus one index date per patient. The heart-rate CTEs, the query
budget and every baseline parameter are settled here, so the SQL this notebook runs and the ceiling
it refuses to spend past sit in one place. There is no analysis in this section.

#### Imports, the Heart-Rate CTEs, the Query Budget, and the Baseline Parameters

The budget exists because the minute-level heart-rate table is the one object in this project large
enough for a careless scan to be expensive, and everything downstream of it is priced by a dry run
before it executes. **The grid reaches back to rel_day `GRID_FIRST` rather than to the reported
span's edge**, because the static baseline's lookback is anchored to the surgery and the trailing
windows to the day they score — which also means the minute-level scan is wider than the figures,
and the budget has to be read against that wider span.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Walk up from the kernel's cwd to the repo root, so this notebook runs from its task
# directory or from the root without the path being hard-coded in either.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import baselines as B
import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD, SURFACE

cfg = srp_common.init("fitbit_heart_rate")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq
reportable = srp_common.reportable_days

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used until then
# because it is the only rule no post-operative event can influence, and every result below is
# stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS
BASELINE_STATISTIC = B.BASELINE_CENTRE   # "median": resting heart rate is mildly right-skewed
REPORT_FIRST, REPORT_LAST = -30, 30      # the span every baseline figure is drawn over
ZOOM = (-10, 10)                         # the admission itself, at readable width
GRID_FIRST = B.grid_first_day(REPORT_FIRST)

# Days a patient needs in EACH window to be analysable. Part 1 counts these patients and Part 2
# derives the resting value for exactly them, so the rule is written once: a second copy of the 7
# is how a curve and the scorecard row behind it drift apart.
PAIRED_MIN_DAYS = 7

_window_case = srp_common.window_case_sql("rel_day")

METRIC_CTE = f"""
    hr_daily AS (
        SELECT
            h.person_id,
            DATE_DIFF(h.date, i.index_date, DAY) AS rel_day,
            SUM(h.minute_in_zone)                AS minutes_recorded
        FROM `{CDR}`.heart_rate_summary AS h
        JOIN index_surgery AS i ON i.person_id = h.person_id
        GROUP BY h.person_id, rel_day
    ),
    hr_window_counts AS (
        SELECT
            person_id,
            {_window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM hr_daily
        WHERE {_window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    hr_window_days AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'postop',   n_days, 0)) AS postop_days
        FROM hr_window_counts
        GROUP BY person_id
    ),
    paired_hr_person AS (
        SELECT person_id
        FROM hr_window_days
        WHERE baseline_days >= {PAIRED_MIN_DAYS}
          AND postop_days   >= {PAIRED_MIN_DAYS}
    )"""

# Deriving a resting value needs the minute-level table, which is orders of magnitude larger.
# Nothing reads it until a dry run has priced the scan and it comes in under this budget -- an
# accidental full scan is the expensive mistake in this repo.
MINUTE_LEVEL_BUDGET_GIB = 200

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

# Fail here, not several minutes into a BigQuery run, if the CDR does not have what this notebook
# reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "heart_rate_intraday": ["person_id", "datetime", "heart_rate_value"],
    "heart_rate_summary": ["person_id", "date", "minute_in_zone"],
})

print(f"index rule           : {INDEX_RULE}")
print(f"baseline statistic   : {BASELINE_STATISTIC}")
print("windows              : "
      + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))
print(f"reported span        : rel_day {REPORT_FIRST:+d} .. {REPORT_LAST:+d}")
print(f"grid spans           : rel_day {GRID_FIRST:+d} .. {REPORT_LAST:+d}"
      f"   ({REPORT_LAST - GRID_FIRST + 1} days of minute-level data)")
print(f"minute-level budget  : {MINUTE_LEVEL_BUDGET_GIB} GiB")
print()
print("the four baselines, at the values inherited from the sweeps")
print(f"  static      : {B.STATIC_LOOKBACK_DAYS}d window ending day "
      f"{-B.STATIC_DAYS_PRIOR:+d}, i.e. rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. "
      f"{-B.STATIC_DAYS_PRIOR:+d}, needing {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days")
print(f"  rolling     : {B.ROLLING_WINDOW_DAYS}d trailing window, gap "
      f"{B.ROLLING_GAP_DAYS}d, needing {B.ROLLING_MIN_DAYS} of its "
      f"{B.ROLLING_WINDOW_DAYS} days ({1 - B.ROLLING_MAX_MISSING:.0%} of them)")
print(f"  ewma        : halflife {B.EWMA_HALFLIFE_DAYS}d, capacity "
      f"{B.ewma_full_effective_days():.1f} effective days, needing "
      f"{B.EWMA_MIN_EFFECTIVE_DAYS:.1f} of them")
print(f"  comparative : {B.SHORT_SPAN_DAYS}d short vs {B.LONG_SPAN_DAYS}d long, "
      f"{B.SPAN_COUPLING}, in units of the long window's IQR")
print()
print(f"small-cell threshold : {SUPPRESS_THRESHOLD} patients per reported day")

## Part 1 — Coverage From the Cheap Table

Coverage is settled entirely from the daily summary, which is small enough to scan across the whole
relative-day range, so the expensive table is never touched to answer a question the cheap one can
answer. **Coverage here is an upper bound on resting heart rate**, which additionally needs enough
minutes in a day to estimate from, and a day counts as covered when the summary records any minutes
at all — generous on purpose, so the bound is honest.

The paired count is what Part 2 derives its resting value over, not merely a number reported here.
Restricting the expensive derivation to the paired set is a correctness choice before it is a cost
one: a signal read over patients who enter and leave between the two windows is a composition
change, which is the failure this whole notebook is arranged to avoid.

#### Who Is Covered, and Who Has Both Windows

The daily summary and the minute-level table are named and shaped differently across All of Us
releases, so both are declared to `require_columns` and validated at `init()` against `cdr_schema.py`
rather than probed at runtime. **The paired count is the binding constraint on any return-to-baseline
construction**, as it is for steps and sleep — neither window alone decides anything.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM hr_daily)    AS patients_with_any_hr,
        (SELECT COUNT(*) FROM hr_daily)                     AS hr_days
"""

cov = cached_gbq(coverage_sql, "heart_rate_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients          : {cohort_patients:>9,}")
print(f"  ... with >=1 HR day    : {int(cov['patients_with_any_hr']):>9,}"
      f"   ({int(cov['patients_with_any_hr']) / cohort_patients:6.1%} of cohort)")
print(f"heart-rate days          : {int(cov['hr_days']):>9,}")

# The denominator curve, from the cheap table. This is a count of patients rather than a reading
# of the signal, so it is not the cross-sectional curve the baselines exist to replace -- it is
# the coverage the baselines will have to be built out of.
curve_sql = CTES + f"""
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM hr_daily
    WHERE rel_day BETWEEN {GRID_FIRST} AND {REPORT_LAST}
    GROUP BY rel_day
    ORDER BY rel_day
"""

hr_curve = cached_gbq(curve_sql, "heart_rate_coverage_curve")

srp_common.plot_rel_day_curve(
    cfg, hr_curve, "n_patients",
    title="Patients with a Fitbit heart-rate record, by day relative to surgery",
    ylabel="patients with a heart-rate record", suppress=True,
    save_as="heart_rate_coverage")

# The paired count: both windows populated for the same patient. This reads the same
# hr_window_days CTE that paired_hr_person is cut from, so the number printed here and the
# population Part 2 derives its resting value over cannot disagree.
paired_sql = CTES + f"""
    SELECT
        COUNT(*)                                     AS patients_any_window,
        COUNTIF(baseline_days >= {PAIRED_MIN_DAYS})  AS baseline_7d,
        COUNTIF(postop_days   >= {PAIRED_MIN_DAYS})  AS postop_7d,
        COUNTIF(baseline_days >= {PAIRED_MIN_DAYS}
            AND postop_days   >= {PAIRED_MIN_DAYS})  AS paired_7d
    FROM hr_window_days
"""

paired = cached_gbq(paired_sql, "heart_rate_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "postop_7d", "paired_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

## Part 2 — What the Derived Signal Costs, and the Person-Day Grid It Buys

The resting value is taken as each patient-day's 5th percentile of minute-level readings, which is
the usual proxy when a device's own resting figure is not published. **The query runs only if a dry
run prices it under the budget set in setup**, so an accidental full scan of the minute-level table
cannot happen here, and a scan that comes in over budget is reported as a finding rather than
quietly paid for.

What the query returns is one row per patient-day, not a per-day aggregate, because a baseline is a
per-patient object and cannot be reconstructed from cohort medians. Those rows are participant-level
and stay in the kernel; the grid built from them is the only thing the rest of the notebook reads,
and every number taken out of it comes through `reportable()`.

Restricting to the paired set prunes bytes scanned only if the table is clustered on `person_id`, so
the dry-run estimate may barely move — the restriction is there for correctness, and any saving is
incidental.

#### Pricing the Derivation, and What the Eligibility Rule Removes

Two bounds sit on a derived resting value and both are judgements about the device rather than about
the patient. **A day needs enough minutes recorded for a 5th percentile to mean anything**, which is
the larger of the two rules by far, and a value outside a plausible physiological band is an artifact
rather than a reading. Each is printed with the person-days it removes rather than buried in the
query.

In [ ]:
# A 5th percentile over a handful of minutes is not a resting value, so a day needs this many
# minutes recorded before it is allowed to be one. This is the rule that does most of the dropping.
MIN_RECORDED_MINUTES = 180

# The plausibility band. Outside it the reading is a device artifact rather than a resting heart
# rate, and neither bound is near anything a surgical patient would plausibly sustain for a day.
MIN_RESTING_HR = 30
MAX_RESTING_HR = 120

extract_sql = CTES + f"""
    , minute_hr AS (
        SELECT
            m.person_id,
            DATE_DIFF(DATE(m.datetime), i.index_date, DAY) AS rel_day,
            m.heart_rate_value
        FROM `{CDR}`.heart_rate_intraday AS m
        JOIN index_surgery    AS i ON i.person_id = m.person_id
        JOIN paired_hr_person AS p ON p.person_id = m.person_id
        WHERE DATE(m.datetime) BETWEEN DATE_SUB(i.index_date, INTERVAL {-GRID_FIRST} DAY)
                                   AND DATE_ADD(i.index_date, INTERVAL {REPORT_LAST} DAY)
    )
    SELECT
        person_id,
        rel_day,
        COUNT(*) AS minutes,
        APPROX_QUANTILES(heart_rate_value, 100)[OFFSET(5)] AS resting_hr
    FROM minute_hr
    WHERE rel_day BETWEEN {GRID_FIRST} AND {REPORT_LAST}
    GROUP BY person_id, rel_day
"""

estimated_bytes = cfg.dry_run_bytes(extract_sql)
grid = None

if estimated_bytes < 0:
    print("Dry run unavailable, so the scan is unpriced and the query was NOT run. Price it by "
          "hand before raising MINUTE_LEVEL_BUDGET_GIB or running it directly.")
elif estimated_bytes > MINUTE_LEVEL_BUDGET_GIB * 2**30:
    print(f"Estimated scan {estimated_bytes / 2**30:,.1f} GiB exceeds the "
          f"{MINUTE_LEVEL_BUDGET_GIB} GiB budget -- narrow the day range or raise the budget "
          "deliberately. The cost itself is an argument against this metric.")
else:
    # NOTHING BELOW PRINTS A ROW. `hr_days` and `grid` are participant-level and stay in the
    # kernel; every number printed or plotted from here on is a count or a quantile.
    hr_days = cached_gbq(extract_sql, "resting_heart_rate_person_days")
    hr_days["eligible"] = (
        hr_days["minutes"].ge(MIN_RECORDED_MINUTES)
        & hr_days["resting_hr"].between(MIN_RESTING_HR, MAX_RESTING_HR)
    )
    grid = B.eligible_value_grid(hr_days, "resting_hr", "eligible", GRID_FIRST, REPORT_LAST)

    recorded = len(hr_days)
    eligible_days = int(grid.notna().to_numpy().sum())
    print()
    print(f"derived patient-days in the grid's span : {recorded:>10,}")
    print(f"  ... passing the eligibility rule      : {eligible_days:>10,}"
          f"   ({eligible_days / recorded:6.1%})")
    print(f"patients with any derived day           : "
          f"{hr_days['person_id'].nunique():>10,}")
    print(f"  ... with any ELIGIBLE day             : {grid.shape[1]:>10,}")
    print(f"grid                                    : "
          f"{grid.shape[0]:,} days x {grid.shape[1]:,} patients"
          f"   ({grid.notna().to_numpy().mean():.1%} filled)")

    print()
    for label, failed in [
        (f"under {MIN_RECORDED_MINUTES} minutes recorded",
         ~hr_days["minutes"].ge(MIN_RECORDED_MINUTES)),
        (f"resting value outside [{MIN_RESTING_HR}, {MAX_RESTING_HR}] bpm",
         ~hr_days["resting_hr"].between(MIN_RESTING_HR, MAX_RESTING_HR)),
    ]:
        n = int(failed.sum())
        print(f"  days failing {label:<44s} {n:>9,}   ({n / recorded:6.2%})")

    coverage = reportable(grid)
    srp_common.plot_rel_day_curve(
        cfg, coverage.loc[coverage["rel_day"].between(GRID_FIRST, REPORT_LAST)], "n_patients",
        title="Patients contributing an eligible resting heart rate, by day relative to surgery",
        ylabel="patients with an eligible derived day", suppress=True, show_windows=True,
        save_as="grid_coverage")

    print()
    print(f"{'day':>6s}{'patients':>12s}{'median resting hr':>20s}")
    for day in [REPORT_FIRST, -14, -7, -1, 1, 7, 14, REPORT_LAST]:
        row = coverage.loc[coverage["rel_day"] == day]
        if row.empty:
            print(f"{day:>+6d}{'below the threshold, not reported':>32s}")
        else:
            print(f"{day:>+6d}{int(row['n_patients'].iloc[0]):>12,}"
                  f"{row['p50'].iloc[0]:>20.1f}")

## Part 3 — What Each Baseline Calls Normal

A baseline is a claim about where a patient's normal sits, so the first thing to look at is the
claims themselves rather than the departures from them. Drawn as levels, all four are in beats per
minute and can share an axis, which is the only figure in this notebook where that is true — and the
observed cross-sectional curve is drawn beside them, dotted, as the thing they exist to replace.

Resting heart rate is the measure where a per-patient baseline matters most, because the spread
between people is far wider than the movement within one. A cohort median sitting at 68 bpm says
nothing about a patient whose own normal is 52, and a five-beat rise means something quite different
for each of them. **The static row of the movement table bounds the composition effect**: each
patient's static level is frozen, so any movement in it is purely the set of contributing patients
changing underneath a fixed set of values.

The coverage table beside it is the price. Every baseline refuses the person-days it cannot serve,
and here that refusal compounds with a derivation that already requires a well-worn device — so a
baseline costing half the analysable cohort has to buy something with it.

#### The Four Levels, and What Each One Costs

The trailing baselines drift toward the post-operative days they are partly built from, so they
under-report a sustained change; the frozen one cannot follow a patient whose normal genuinely moved.
**Neither failure is fixable by tuning, which is why all four are carried rather than one chosen
here.** The per-day scored counts show that the cost is not constant across the span — a trailing
baseline is hungriest exactly where the device is least likely to have been worn.

In [ ]:
if grid is None:
    raise RuntimeError(
        "No person-day grid: the minute-level derivation in Part 2 was skipped, either because "
        "the dry run was unavailable or because the estimate exceeded MINUTE_LEVEL_BUDGET_GIB. "
        "Nothing below can be computed until that query runs."
    )

# Computed once and reused by Parts 4 and 5: the rolling medians over a grid this size are the
# expensive step outside BigQuery, and nothing below them depends on anything else.
levels = B.baseline_levels(grid, statistic=BASELINE_STATISTIC)
deviations = B.deviations_from_each_baseline(grid, statistic=BASELINE_STATISTIC)
level_curves = {b: reportable(levels[b]) for b in B.BASELINE_NAMES}
departure_curves = {b: reportable(deviations[b]) for b in B.BASELINE_NAMES}

# THE OBSERVED CURVE IS CROSS-SECTIONAL -- it moves when the set of contributing patients moves,
# which is the problem the baselines exist to remove, so it is drawn as context, not as a result.
srp_common.plot_baseline_curves(
    cfg, {"observed day": coverage,
          **{f"{b} baseline": level_curves[b] for b in B.BASELINE_NAMES}},
    title="Where each baseline puts a patient's normal resting heart rate, "
          "against the day actually recorded",
    ylabel="resting heart rate (bpm)", save_as="baseline_levels",
    span=(REPORT_FIRST, REPORT_LAST), colours=[INK_MUTED, *CATEGORICAL],
    note="median across contributing patients; levels only, not departures")

print("median level across contributing patients, by window (bpm)")
print(f"{'window':<22s}{'observed':>12s}" + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 7, "first post-op week"), (8, 30, "rest of the month")]:
    print(f"{label:<22s}", end="")
    window = coverage.loc[coverage["rel_day"].between(lo, hi), "p50"]
    print(f"{window.median():>12.1f}" if len(window) else f"{'--':>12s}", end="")
    for b in B.BASELINE_NAMES:
        frame = level_curves[b]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        print(f"{window.median():>14.1f}" if len(window) else f"{'--':>14s}", end="")
    print()

print()
print("movement of the baseline itself, first post-op week vs the fortnight before")
for b in B.BASELINE_NAMES:
    frame = level_curves[b]
    before = frame.loc[frame["rel_day"].between(-14, -1), "p50"].median()
    after = frame.loc[frame["rel_day"].between(1, 7), "p50"].median()
    if pd.isna(before) or pd.isna(after):
        print(f"  {b:<12s} not reportable in one of the two windows")
        continue
    print(f"  {b:<12s} {before:>7.1f} -> {after:>7.1f}   ({after - before:+6.2f} bpm)")

cost = B.coverage_by_baseline(grid, deviations)

print()
print(f"{int(grid.notna().to_numpy().sum()):,} eligible person-days over "
      f"{grid.shape[1]:,} patients")
print(f"{'baseline':<14s}{'person-days':>13s}{'of eligible':>13s}"
      f"{'patients':>11s}{'of eligible':>13s}")
for _, row in cost.iterrows():
    flag = ("   << below the small-cell threshold"
            if 0 < row["patients"] < SUPPRESS_THRESHOLD else "")
    print(f"{row['baseline']:<14s}{int(row['person_days']):>13,}"
          f"{row['share_of_person_days']:>12.1%}"
          f"{int(row['patients']):>11,}{row['share_of_patients']:>12.1%}{flag}")

print()
print("the floor doing the dropping, in each case")
print(f"  static      : {B.STATIC_MIN_ELIGIBLE_DAYS} eligible days inside rel_day "
      f"{-(B.STATIC_DAYS_PRIOR + B.STATIC_LOOKBACK_DAYS - 1):+d} .. {-B.STATIC_DAYS_PRIOR:+d}"
      f"   ({B.STATIC_MIN_ELIGIBLE_DAYS / B.STATIC_LOOKBACK_DAYS:.0%} of that window)")
print(f"  rolling     : {B.ROLLING_MIN_DAYS} eligible days in the trailing "
      f"{B.ROLLING_WINDOW_DAYS} days")
print(f"  ewma        : {B.EWMA_MIN_EFFECTIVE_DAYS:.1f} effective days out of a capacity of "
      f"{B.ewma_full_effective_days():.1f}")
print(f"  comparative : {B.days_required(B.SHORT_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} "
      f"days in the short window AND "
      f"{B.days_required(B.LONG_SPAN_DAYS, B.ROLLING_MAX_MISSING, floor=2)} in the long one, "
      f"the long one also needing a non-zero IQR")

srp_common.plot_baseline_coverage(
    cfg, grid, deviations, (REPORT_FIRST, REPORT_LAST),
    title="Patients each baseline can actually score, by day relative to surgery",
    save_as="baseline_coverage")

print()
print(f"thinnest reported day for each baseline, within rel_day {REPORT_FIRST:+d} .. "
      f"{REPORT_LAST:+d}")
for b in B.BASELINE_NAMES:
    scored = deviations[b].notna().sum(axis=1).loc[REPORT_FIRST:REPORT_LAST]
    below = int((scored < SUPPRESS_THRESHOLD).sum())
    print(f"  {b:<12s} {int(scored.min()):>7,} patients on day {int(scored.idxmin()):+d}"
          f"   ({below} of {len(scored)} days below the threshold)")

# Only read in this cell. Dropping them keeps a grid's worth of baselines out of the kernel for
# the rest of the notebook.
del levels

## Part 4 — Departure From Normal Around Surgery

This is what the baselines are for: how far each day's resting value sat from that patient's own
normal, which is the quantity a recovery metric would actually be built on. Differencing against a
per-patient level removes the composition effect a cross-sectional curve cannot, because a patient
can only enter a day's median by contributing a day **and** having a baseline of their own to be read
against.

**Three of the four departures share a unit and the fourth does not.** Static, rolling and
exponential are a day's resting value minus a level, so they are in beats per minute; comparative is
a difference of two window centres divided by a spread, so it is in units of that patient's own
long-window IQR. They are drawn on separate axes for that reason.

**The sign here runs the opposite way to steps and sleep.** A raised resting heart rate is the sick
direction, so a departure ABOVE zero is the post-operative signal, and a curve returning to zero from
above is the recovery. That is worth stating once rather than leaving a reader to infer it from a
figure that looks upside down beside the other two notebooks.

#### The Departure Curves, the Peak, and How Long It Takes to Settle

The return question is asked on the departure curve rather than on a cohort median, because a cohort
curve returning to its old level says nothing about whether the patients on it are the same patients.
**The settle target is a fraction of a beat rather than an exact return**, because an exact return to
a noisy baseline is noise-chasing, and a sustained rule is used so that one good day is not a
recovery.

In [ ]:
UNIT_BASELINES = ("static", "rolling", "ewma")   # the three in bpm
SETTLE_BPM = 1.0    # within a beat of normal counts as settled; an exact return is noise-chasing
SUSTAIN_DAYS = 7    # days the median must hold it before it counts as sustained

srp_common.plot_baseline_curves(
    cfg, {f"{b} baseline": departure_curves[b] for b in UNIT_BASELINES},
    title="How far each day's resting heart rate sat from that patient's own normal, "
          "by day relative to surgery",
    ylabel="resting heart rate - that patient's baseline (bpm)",
    save_as="departures", zero_line=True, span=(REPORT_FIRST, REPORT_LAST),
    note="median across contributing patients; above zero is the raised, post-operative direction")

srp_common.plot_baseline_curves(
    cfg, {f"{b} baseline": departure_curves[b] for b in UNIT_BASELINES},
    title="The same three departures through the ten days either side of surgery",
    ylabel="resting heart rate - that patient's baseline (bpm)",
    save_as="departures_zoom", zero_line=True, span=ZOOM,
    note="median across contributing patients; above zero is the raised, post-operative direction")

# Drawn alone because its unit is not the others'. THE Y AXIS HERE IS IQRs OF THAT PATIENT'S OWN
# LONG WINDOW, not bpm.
srp_common.plot_baseline_curves(
    cfg, {"comparative index": departure_curves["comparative"]},
    title="The recent fortnight of resting heart rate against the established normal, "
          "in that patient's own IQRs",
    ylabel="(short centre - long centre) / long IQR",
    save_as="comparative_index", zero_line=True, span=(REPORT_FIRST, REPORT_LAST),
    note="median across contributing patients; unitless, not comparable to the figures above")

print("median departure from baseline, by window")
print(f"{'window':<24s}" + "".join(f"{b:>14s}" for b in B.BASELINE_NAMES))
for lo, hi, label in [(-30, -15, "baseline, first half"), (-14, -1, "the fortnight before"),
                      (1, 3, "first three days"), (4, 7, "rest of week one"),
                      (8, 14, "week two"), (15, 30, "weeks three and four")]:
    print(f"{label:<24s}", end="")
    for b in B.BASELINE_NAMES:
        frame = departure_curves[b]
        window = frame.loc[frame["rel_day"].between(lo, hi), "p50"]
        print(f"{window.median():>+14.3f}" if len(window) else f"{'--':>14s}", end="")
    print()
print("  (the first three columns are bpm; comparative is long-window IQRs)")

# The peak and the settle, read off the WITHIN-PATIENT curve. The static baseline is the one used:
# it is the only one of the four that does not drift toward the post-operative days, so it is the
# only one on which "settled" means back to where the patient was rather than to where they are now.
static_departures = departure_curves["static"]
post = static_departures.loc[static_departures["rel_day"] >= 0].reset_index(drop=True)
peak = post.loc[post["p50"].idxmax()]

print()
print(f"post-operative peak departure : {peak['p50']:>+7.3f} bpm on day "
      f"{int(peak['rel_day']):+d}   ({int(peak['n_patients']):,} patients)")
print(f"settle target                 : within {SETTLE_BPM:.1f} bpm of normal, held "
      f"{SUSTAIN_DAYS} days")

within = (post["p50"].abs() <= SETTLE_BPM).astype(int)
sustained = within.rolling(SUSTAIN_DAYS).sum() == SUSTAIN_DAYS
if sustained.any():
    # idxmax gives the row the window ENDS on; the run starts SUSTAIN_DAYS-1 rows earlier.
    # Indexing back by position rather than subtracting days keeps this correct where the curve
    # has gaps, which it will wherever a day fell below the suppression threshold.
    median_settled_on_day = int(post.loc[sustained.idxmax() - SUSTAIN_DAYS + 1, "rel_day"])
    print(f"median departure first settles : day {median_settled_on_day:+d}")
else:
    median_settled_on_day = None
    print(f"median departure never settles within day {int(post['rel_day'].max())} -- on this "
          "baseline the median patient's resting rate has not returned to normal by the end of "
          "the reported span")

print()
print(f"{'day':>5s}" + "".join(f"{b:>14s}{'n':>9s}" for b in UNIT_BASELINES))
for day in range(ZOOM[0], ZOOM[1] + 1):
    print(f"{day:>+5d}", end="")
    for b in UNIT_BASELINES:
        row = departure_curves[b].loc[departure_curves[b]["rel_day"] == day]
        if row.empty:
            print(f"{'--':>14s}{'--':>9s}", end="")
        else:
            print(f"{row['p50'].iloc[0]:>+14.3f}{int(row['n_patients'].iloc[0]):>9,}", end="")
    print()

## Part 5 — Whether the Choice of Baseline Changes the Answer

Two baselines that rank person-days the same way are interchangeable for anything downstream that
only cares about which days were worse than usual, whatever their levels look like side by side.
Spearman is the right coefficient because the comparative index is on a different scale entirely, so
only the **ordering** of person-days is comparable across a pair. Each pair is computed on the
person-days both baselines score, so the counts differ from the coverage table in Part 3.

Rank correlation says nothing about **where zero is**, and zero is the whole claim — "this day was
further from normal than that patient's usual". Two baselines can rank a month identically and still
put a third of it on opposite sides of normal, and a metric counting abnormal days would then count
different days. Both questions are asked on the post-operative month as well as the whole grid,
because a pair can agree across a year of ordinary days and disagree exactly where the surgery is.

#### Rank Agreement, Side of Normal, and the Share Called Raised

The two thresholds are conventions and are named in the cell rather than left to the reader's
judgement. **The last table counts days ABOVE normal rather than below**, because for resting heart
rate the raised direction is the sick one — and a baseline calling most of the post-operative month
raised is mis-centred rather than sensitive.

In [ ]:
# Above INTERCHANGEABLE the pair ranks days the same and either will do; below DISTINCT the two
# are answering different questions and the choice between them IS the analysis.
INTERCHANGEABLE = 0.95
DISTINCT = 0.70
POSTOP = WINDOWS["postop"]

postop_only = {b: dev.loc[POSTOP[0]:POSTOP[1]] for b, dev in deviations.items()}
whole = B.baseline_agreement(deviations)
post_agreement = B.baseline_agreement(postop_only)

print("pairwise Spearman on the person-days both baselines score")
print(f"{'pair':<28s}{'whole grid':>13s}{'n':>14s}{'postop only':>14s}{'n':>14s}")
for (_, w), (_, p) in zip(whole.iterrows(), post_agreement.iterrows()):
    print(f"{w['pair']:<28s}{w['spearman']:>13.3f}{int(w['person_days']):>14,}"
          f"{p['spearman']:>14.3f}{int(p['person_days']):>14,}")

print()
print(f"reading, on rel_day {POSTOP[0]:+d} .. {POSTOP[1]:+d}")
for _, row in post_agreement.iterrows():
    rho = row["spearman"]
    if pd.isna(rho):
        verdict = "not enough shared person-days to say"
    elif rho >= INTERCHANGEABLE:
        verdict = "interchangeable for anything that only ranks days"
    elif rho >= DISTINCT:
        verdict = "related but not substitutable"
    else:
        verdict = "different questions; the choice between them IS the analysis"
    print(f"  {row['pair']:<28s} {rho:>6.3f}   {verdict}")

print()
print(f"share of shared person-days the pair puts on the SAME side of normal, rel_day "
      f"{POSTOP[0]:+d} .. {POSTOP[1]:+d}")
print(f"{'pair':<28s}{'person-days':>14s}{'same side':>12s}{'both raised':>13s}")
for i, left in enumerate(B.BASELINE_NAMES):
    for right in B.BASELINE_NAMES[i + 1:]:
        a, b = B.flatten_pair(postop_only[left], postop_only[right])
        if len(a) < SUPPRESS_THRESHOLD:
            print(f"{left + ' vs ' + right:<28s}{len(a):>14,}"
                  f"{'--':>12s}{'--':>13s}   too few shared person-days")
            continue
        print(f"{left + ' vs ' + right:<28s}{int(len(a)):>14,}"
              f"{float(((a > 0) == (b > 0)).mean()):>12.1%}"
              f"{float(((a > 0) & (b > 0)).mean()):>13.1%}")

print()
print("share of scored post-operative person-days each baseline calls raised above normal")
for b in B.BASELINE_NAMES:
    values = B.flatten(postop_only[b])
    if len(values) < SUPPRESS_THRESHOLD:
        print(f"  {b:<12s} too few scored person-days to report")
        continue
    print(f"  {b:<12s} {float((values > 0).mean()):>6.1%}   of {len(values):,} person-days")

## Part 6 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Writing the verdict as data rather than prose is what lets the cross-metric comparison be computed
instead of transcribed. **Cost is recorded as a field rather than as a note**, because for this
metric alone the price of the derivation is a selection criterion — a metric that cannot be computed
inside the query budget is not a candidate however well it performs on the days it does cover.

In [ ]:
scored = {b: int(cost.set_index("baseline").loc[b, "patients"]) for b in B.BASELINE_NAMES}

srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="heart_rate_summary + heart_rate_intraday",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    outcome_definition=(
        f"daily resting heart rate, the 5th percentile of minute-level readings on days with "
        f">= {MIN_RECORDED_MINUTES} minutes recorded, read against that patient's own baseline; "
        "four baselines compared, none yet settled"
    ),
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_hr"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    derivation_priced_gib=(estimated_bytes / 2**30 if estimated_bytes >= 0 else None),
    patients_with_eligible_day=int(grid.shape[1]),
    baseline_choice="unsettled",
    patients_scored_by_baseline="; ".join(f"{b}={scored[b]:,}" for b in B.BASELINE_NAMES),
    median_departure_settled_on_day=median_settled_on_day,
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Read within patient against baselines.py, not cross-sectionally. The resting value is "
           "derived, not published: the 5th percentile of minute-level readings on days with at "
           "least the minimum minutes recorded. Coverage is from the daily summary and is "
           "therefore an upper bound on the derived metric. Baseline parameters are inherited "
           "from a cancer-diagnosis sweep and have not been re-swept against this cohort."),
)